# nanochst: Supervised Fine-Tuning 
## Yoav Ram

This notebook continues from `nanochat.ipynb`.
We take the *pretrained* language model and teach it to follow instructions via *supervised fine-tuning* or **SFT**.

Prerequisite: run `nanochat.ipynb` first — it saves `nanochat_checkpoint.pkl`.

In [ ]:
# ── All model definitions (identical to nanochat.ipynb) ──────────────────
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pickle
import math
import re
import urllib.request
import optax
from functools import partial

from bpe import bpe_encode, bpe_decode, bpe_load

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────
def rms_norm(g, x, eps=1e-6):
    return g * x / jnp.sqrt(jnp.mean(x**2, axis=-1, keepdims=True) + eps)

def precompute_rope(seq_len, head_dim, base=10000.):
    i = jnp.arange(0, head_dim, 2)
    angles = jnp.outer(jnp.arange(seq_len), 1.0 / (base ** (i / head_dim)))
    angles = jnp.concatenate([angles, angles], axis=-1)
    return jnp.cos(angles), jnp.sin(angles)

def apply_rope(x, cos, sin):
    d = x.shape[-1] // 2
    return x * cos + jnp.concatenate([-x[..., d:], x[..., :d]], axis=-1) * sin

def causal_mask(T):
    return jnp.where(jnp.tril(jnp.ones((T, T))), 0., -jnp.inf)

def attention_forward(p, x, cos, sin, mask):
    B, T, d = x.shape; H = p['n_heads']; hd = d // H
    Q, K, V = x @ p['Wq'], x @ p['Wk'], x @ p['Wv']
    def sh(t): return t.reshape(B, T, H, hd).transpose(0, 2, 1, 3)
    Q, K, V = sh(Q), sh(K), sh(V)
    c, s = cos[None, None], sin[None, None]
    Q = apply_rope(Q, c, s) / (jnp.linalg.norm(Q, axis=-1, keepdims=True) + 1e-6)
    K = apply_rope(K, c, s) / (jnp.linalg.norm(K, axis=-1, keepdims=True) + 1e-6)
    w = jax.nn.softmax(Q @ K.transpose(0, 1, 3, 2) / math.sqrt(hd) + mask[None, None], axis=-1)
    return (w @ V).transpose(0, 2, 1, 3).reshape(B, T, d) @ p['Wo'], w

def mlp_forward(p, x):
    return jax.nn.relu(x @ p['W1']) ** 2 @ p['W2']

def forward(params, ids, cos, sin, mask):
    x = params['tok_emb']['W'][ids]
    for blk in params['blocks']:
        x = x + attention_forward(blk['attn'], rms_norm(blk['norm1']['g'], x), cos, sin, mask)[0]
        x = x + mlp_forward(blk['mlp'], rms_norm(blk['norm2']['g'], x))
    return rms_norm(params['norm_f']['g'], x) @ params['head']['W'].T

def sample_token(key, logits, temperature=1.0, top_k=0, top_p=1.0):
    logits = logits / temperature
    if top_k > 0:
        kth = jnp.sort(logits)[-top_k]
        logits = jnp.where(logits < kth, -jnp.inf, logits)
    probs = jax.nn.softmax(logits)
    if top_p < 1.0:
        sorted_idx = jnp.argsort(-probs)
        cumsum     = jnp.cumsum(probs[sorted_idx])
        keep  = jnp.concatenate([jnp.array([True]), cumsum[:-1] <= top_p])
        mask  = jnp.zeros_like(probs).at[sorted_idx].set(keep.astype(probs.dtype))
        probs = probs * mask
        probs = probs / (probs.sum() + 1e-9)
    return jax.random.choice(key, probs.shape[0], p=probs)

def generate(params, vocab, merges, prompt, cfg,
             max_new_tokens=80, temperature=1.0, top_k=0, top_p=1.0, key=None):
    if key is None: key = jax.random.PRNGKey(0)
    seq_len = cfg.get('seq_len', 128)
    ids = bpe_encode(prompt, vocab, merges)
    for _ in range(max_new_tokens):
        ctx = ids[-seq_len:]; T = len(ctx)
        cos, sin = precompute_rope(T, cfg['head_dim'])
        logits = forward(params, jnp.array([ctx]), cos, sin, causal_mask(T))[0, -1]
        key, sub = jax.random.split(key)
        ids.append(int(sample_token(sub, logits, temperature, top_k, top_p)))
    return bpe_decode(ids, vocab)

def save_checkpoint(params, cfg, vocab, merges, path):
    with open(path, 'wb') as f:
        pickle.dump({'params': jax.tree_util.tree_map(np.array, params),
                     'cfg': cfg, 'vocab': vocab, 'merges': merges}, f)
    print(f'Saved -> {path}')

def load_checkpoint(path):
    with open(path, 'rb') as f: s = pickle.load(f)
    vocab  = s['vocab']
    merges = [tuple(m) for m in s['merges']]
    return jax.tree_util.tree_map(jnp.array, s['params']), s['cfg'], vocab, merges

print(f'JAX {jax.__version__} | {jax.default_backend()}')

In [ ]:
CHECKPOINT = 'nanochat_checkpoint.pkl'

pretrained_params, cfg, vocab, merges = load_checkpoint(CHECKPOINT)

SEQ_LEN = cfg.get('seq_len', 128)
cos_c, sin_c = precompute_rope(SEQ_LEN, cfg['head_dim'])
mask_c       = causal_mask(SEQ_LEN)

print(f'Loaded model: {cfg}')
print(f'Vocab size:   {len(vocab)}')

print('\nPretrained sample (no instruction):')
print(generate(pretrained_params, vocab, merges, 'Once upon a time', cfg,
               max_new_tokens=40, temperature=0.8, key=jax.random.PRNGKey(1)))

## What is Supervised Fine-Tuning - SFT?

After pretraining, the model can complete text — it predicts the next token.
It has no concept of a "question" or "instruction". 
Given the prompt `Write a story about a dog.` it might output more instructions rather than a story.

SFT teaches the model a **format**: given a prompt, produce a specific type of response.
This is done by training on `(prompt, response)` pairs, masking the loss on the prompt
so the model only learns to predict the response.

```
Pretraining batch:
  tokens:  [t₁  t₂  t₃  t₄  t₅  t₆  t₇  t₈]
  loss on:  [✓   ✓   ✓   ✓   ✓   ✓   ✓   ✓ ]

SFT batch:
  tokens:  [PROMPT TOKENS  |  RESPONSE TOKENS      ]
  loss on:  [✗   ✗   ✗   ✗  |  ✓    ✓    ✓    ✓   ]
```

The cross-entropy loss is **identical** — we just zero-out the prompt positions:

$$\mathcal{L}_{\text{SFT}} = -\frac{1}{R} \sum_{t=P+1}^{R} \log P_\theta(x_t \mid x_{<t})$$

where tokens $1,\ldots,P$ are the prompt and tokens $P+1,\ldots,R$ are the response.

Why mask the prompt?
If we included the prompt in the loss, the model would spend capacity memorising
the instruction format (which is fixed) rather than learning to produce good responses to specific prompts.

## SFT Dataset

We build `(prompt, response)` pairs directly from TinyStories:

- **Prompt**: `Continue the story: {first sentence}`
- **Response**: `{remainder of the story}`

This requires no new data and gives a clear demonstration:
before SFT, the model ignores the instruction prefix;
after SFT, it reliably produces a story continuation.

In [2]:
CACHE = "data/TinyStoriesV2-GPT4-valid.txt"
with open(CACHE, encoding='utf-8') as f:
    raw = f.read()
    
MAX_STORIES = 100_000
stories = [s.strip() for s in raw.split('<|endoftext|>') if s.strip()][:MAX_STORIES]
text    = "\n".join(stories)
print(f"Stories: {len(stories):,}  |  Characters: {len(text):,}")

Stories: 27,630  |  Characters: 22,095,533


### Chat template

We use a minimal template with explicit role markers:

```
[INST] Continue the story: {first sentence} [/INST] {rest of story}
```

The tokens `[INST]` and `[/INST]` are not special tokens in our BPE vocab —
they will be tokenised as subword pieces. That's fine for this demonstration.
In production models, special tokens are added to the vocabulary and always
tokenised as a single unit (like GPT-2's `<|endoftext|>`).

In [3]:
def make_sft_example(story, vocab, merges, seq_len):
    """
    Split story into prompt (first sentence) + response (rest).
    Returns (input_ids, target_ids, response_mask) or None if story is too short.
    """
    dot = story.find('. ')
    if dot == -1 or dot > len(story) * 0.7:
        return None

    first_sent = story[:dot + 2]
    rest       = story[dot + 2:]
    if len(rest) < 20:
        return None

    prompt   = f'[INST] Continue the story: {first_sent}[/INST] '
    response = rest

    p_ids = bpe_encode(prompt, vocab, merges)
    r_ids = bpe_encode(response, vocab, merges)

    combined = (p_ids + r_ids)[:seq_len + 1]
    if len(combined) < seq_len + 1:
        return None

    inputs  = np.array(combined[:seq_len], dtype=np.int32)
    targets = np.array(combined[1:seq_len + 1], dtype=np.int32)

    resp_start = min(len(p_ids), seq_len)
    mask = np.zeros(seq_len, dtype=np.float32)
    mask[resp_start:] = 1.0

    return inputs, targets, mask

In [4]:
sft_examples = []
for story in stories:
    ex = make_sft_example(story, vocab, merges, SEQ_LEN)
    if ex is not None:
        sft_examples.append(ex)

print(f'SFT examples created: {len(sft_examples):,}')

inputs, targets, resp_mask = sft_examples[0]
print(f'\nExample (first 30 tokens):')
print(f'  Input:  {bpe_decode(inputs[:30].tolist(), vocab)!r}')
print(f'  Target: {bpe_decode(targets[:30].tolist(), vocab)!r}')
print(f'  Mask:   {resp_mask[:30]}')
print(f'  Response starts at position: {int(resp_mask.argmax())}')

NameError: name 'vocab' is not defined

---
## 3. SFT Loss

The only code change from pretraining: multiply the per-token loss by `response_mask`
and normalise by the number of response tokens (not total tokens).

```python
# Pretraining loss:
loss = -mean(log_probs[correct_token])            # over ALL positions

# SFT loss:
loss = -sum(log_probs[correct_token] * mask)      # only RESPONSE positions
      / sum(mask)
```

The gradient flows only through the response positions —
the prompt positions have zero loss and contribute nothing to the parameter update.

In [ ]:
def sft_loss_fn(params, inputs, targets, response_mask):
    """
    inputs         : (B, T) int
    targets        : (B, T) int   — next-token targets
    response_mask  : (B, T) float — 1.0 at response positions, 0.0 at prompt
    """
    B, T = inputs.shape
    logits   = forward(params, inputs, cos_c[:T], sin_c[:T], mask_c[:T, :T])
    log_probs = jax.nn.log_softmax(logits, axis=-1)           # (B, T, V)
    # Log-prob of the actual next token at each position
    correct   = log_probs[jnp.arange(B)[:, None],
                          jnp.arange(T)[None, :],
                          targets]                             # (B, T)
    # Masked mean: loss only on response tokens
    loss = -(correct * response_mask).sum() / (response_mask.sum() + 1e-8)
    return loss

LR_SFT    = 3e-5
optimizer = optax.adamw(LR_SFT, weight_decay=0.1)

@jax.jit
def sft_train_step(params, opt_state, inputs, targets, response_mask):
    loss, grads = jax.value_and_grad(sft_loss_fn)(params, inputs, targets, response_mask)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


# Verify: loss is lower for the correct response than for a random one
inputs_ex = jnp.array([sft_examples[0][0]])
targets_ex = jnp.array([sft_examples[0][1]])
mask_ex    = jnp.array([sft_examples[0][2]])
print(f'SFT loss (pretrained model): {float(sft_loss_fn(pretrained_params, inputs_ex, targets_ex, mask_ex)):.4f}')

---
## 4. SFT Training

The training loop is identical to pretraining with two changes:
1. Batches come from the SFT dataset (prompt+response pairs)
2. We pass `response_mask` to the loss function

We use a **smaller learning rate** (3e-5 vs 3e-4) to avoid catastrophic forgetting:
we want to refine the model's behaviour, not overwrite its world knowledge.

In [ ]:
BATCH_SIZE = 8
N_STEPS    = 200

# Start SFT from the pretrained weights
params_sft = pretrained_params
opt_state  = optimizer.init(params_sft)

key = jax.random.PRNGKey(0)
sft_losses, steps_log = [], []

print(f'SFT training: {N_STEPS} steps, batch={BATCH_SIZE}, lr={LR_SFT}')
print(f'{"Step":>6}  {"SFT loss":>10}')
print('-' * 22)

for step in range(N_STEPS):
    key, subkey = jax.random.split(key)
    idx    = jax.random.randint(subkey, (BATCH_SIZE,), 0, len(sft_examples))
    batch  = [sft_examples[int(i)] for i in idx]
    inputs = jnp.array([b[0] for b in batch])
    targets= jnp.array([b[1] for b in batch])
    masks  = jnp.array([b[2] for b in batch])

    params_sft, opt_state, loss = sft_train_step(params_sft, opt_state, inputs, targets, masks)

    if step % 40 == 0 or step == N_STEPS - 1:
        sft_losses.append(float(loss))
        steps_log.append(step)
        print(f'{step:>6}  {float(loss):>10.4f}')

print('Done.')

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(steps_log, sft_losses, marker='o')
plt.xlabel('Step'); plt.ylabel('SFT loss'); plt.title('SFT training curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Evaluation: Pretrained vs SFT

We prompt both models with the instruction format and compare their outputs.
The pretrained model was never trained to respond to `[INST]...[/INST]`,
so it will likely ignore the instruction and continue free-form.
The SFT model has learned to produce a story continuation after `[/INST]`.

In [ ]:
test_prompts = [
    'Once upon a time there was a little cat named Mia. ',
    'One sunny day, a boy named Tom went to the park. ',
]

for first_sent in test_prompts:
    prompt = f'[INST] Continue the story: {first_sent}[/INST] '
    print(f'Prompt: {first_sent!r}')
    print(f'\nPretrained:')
    print(generate(pretrained_params, vocab, merges, prompt, cfg,
                   max_new_tokens=50, temperature=0.8, key=jax.random.PRNGKey(42)))
    print(f'\nAfter SFT:')
    print(generate(params_sft, vocab, merges, prompt, cfg,
                   max_new_tokens=50, temperature=0.8, key=jax.random.PRNGKey(42)))
    print('\n' + '='*70 + '\n')

In [ ]:
save_checkpoint(params_sft, cfg, vocab, merges, 'nanochat_sft_checkpoint.pkl')

---
## 6. Exercises

### Exercise 1: Prompt ablation
Train two SFT models: one with the `[INST]...[/INST]` template, one without
(just concatenate prompt and response). Do both learn to follow instructions?
Which generalises better to unseen prompts?

### Exercise 2: Learning rate sensitivity
Try SFT learning rates in `{1e-5, 3e-5, 1e-4, 3e-4}`. Plot SFT loss curves.
At what LR does catastrophic forgetting appear — i.e., the model stops producing
coherent text on non-instruction prompts?

### Exercise 3: Prompt-only loss ★
Currently the mask is `1` for response positions, `0` for prompt positions.
Instead, try `response_mask = 1 - prompt_mask` (train on prompts, not responses).
What does the model learn? What does this tell you about what SFT is really doing?

### Exercise 4: Dataset size ablation ★
Train SFT models on 100, 500, 1000, 3000 examples.
At what dataset size does the model reliably respond in the expected format?

### Exercise 5: Biological SFT ★★
Replace TinyStories with protein sequence data.
Create `(prompt, response)` pairs where the prompt is a protein family name
and the response is a sequence in FASTA format.
Does SFT teach the model to generate plausible sequences for a given family?

---
## References

1. **Karpathy (2025)** nanochat. [github.com/karpathy/nanochat](https://github.com/karpathy/nanochat)
2. **Wei et al. (2022)** Finetuned Language Models are Zero-Shot Learners (FLAN). *ICLR 2022*.
3. **Ouyang et al. (2022)** Training language models to follow instructions with human feedback (InstructGPT). *NeurIPS 2022*.
4. **Eldan & Li (2023)** TinyStories. arXiv:2305.07759.
5. **Bradbury et al. (2018)** JAX. [github.com/google/jax](https://github.com/google/jax).